In [60]:
#importing the libraries
import pandas as pd
import numpy as np
df = pd.read_csv('twcs.csv')

**tweet_id:** Unique numerical identifier for the tweet.  
**author_id:** The sender's handle.  
**inbound:** Direction of the message.  
**created_at:** Timestamp when the tweet was posted.  
**text:** The raw tweet body.  
**response_tweet_id:** The ID(s) of the reply tweets.  
**in_response_to_tweet_id:** The parent tweet_id.

In [61]:
df.head(10)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3.0
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1.0
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4.0
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5.0
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6.0
5,6,sprintcare,False,Tue Oct 31 21:46:24 +0000 2017,@115712 Can you please send us a private messa...,"5,7",8.0
6,8,115712,True,Tue Oct 31 21:45:10 +0000 2017,@sprintcare is the worst customer service,"9,6,10",NaN
7,11,sprintcare,False,Tue Oct 31 22:10:35 +0000 2017,@115713 This is saddening to hear. Please shoo...,NaN,12.0
8,12,115713,True,Tue Oct 31 22:04:47 +0000 2017,@sprintcare You gonna magically change your co...,"11,13,14",15.0
9,15,sprintcare,False,Tue Oct 31 20:03:31 +0000 2017,@115713 We understand your concerns and we'd l...,12,16.0


In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2811774 entries, 0 to 2811773
Data columns (total 7 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   tweet_id                 int64  
 1   author_id                object 
 2   inbound                  bool   
 3   created_at               object 
 4   text                     object 
 5   response_tweet_id        object 
 6   in_response_to_tweet_id  float64
dtypes: bool(1), float64(1), int64(1), object(4)
memory usage: 131.4+ MB


In [63]:
df.duplicated().sum()

0

Keeping only AmazonHelp data and dropping the rest

In [64]:
amazon_replies = df[df["author_id"]=="AmazonHelp"]
customer_ids = amazon_replies["in_response_to_tweet_id"].dropna()
target_ids = set(amazon_replies["tweet_id"]).union(set(customer_ids))
amazon_df = df[df["tweet_id"].isin(target_ids)].drop_duplicates(subset=["tweet_id"])
amazon_df.to_csv("amazon_tweets.csv", index=False)


In [65]:
amazon_df.head(25)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
181,269,AmazonHelp,False,Wed Nov 22 09:23:01 +0000 2017,@115770 こんにちは、アマゾン公式です。Fire TV Stickが見れないというのは...,"270,271",272.0
183,271,115770,True,Wed Nov 22 09:30:36 +0000 2017,@AmazonHelp 電話で対応してもらいましたが改良されませんでした。\n保証期間も過ぎ...,273,269.0
184,273,AmazonHelp,False,Wed Nov 22 09:40:27 +0000 2017,@115770 カスタマーサービスにてお問い合わせ済みとのことで、お手数をおかけいたしました...,274,271.0
185,274,115770,True,Wed Nov 22 09:44:04 +0000 2017,@AmazonHelp こちらこそありがとうございました。,275,273.0
186,275,AmazonHelp,False,Wed Nov 22 10:06:26 +0000 2017,@115770 恐れ入ります。至らない点も多々あるかとは存じますが、今後ともどうぞよろしくお...,NaN,274.0
187,272,115770,True,Wed Nov 22 09:14:39 +0000 2017,amazonのfireTVstickが見れない😢,269,NaN
234,324,AmazonHelp,False,Wed Nov 22 09:06:00 +0000 2017,@115792 ご不便をおかけしております。アプリをご利用でしょうか。強制停止&gt;端末の...,NaN,325.0
235,325,115792,True,Wed Nov 22 08:55:35 +0000 2017,amazonプライムビデオ、再生エラーが多いです,324,NaN
321,615,AmazonHelp,False,Tue Oct 31 22:29:00 +0000 2017,@115820 I'm sorry we've let you down! Without ...,616,617.0
322,616,115820,True,Tue Oct 31 23:22:08 +0000 2017,@AmazonHelp 3 different people have given 3 di...,618,615.0


Basic Pre Processing:
1. Removing Emojis
2. lowering
3. stopword removal
4. removing punctuation
5. tokenization
6. stemming

In [66]:
import html
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))

NON_LATIN_PATTERN = re.compile(r"[\u0400-\u04FF\u0600-\u06FF\u2E80-\u9FFF]")


def is_strictly_latin(text: str) -> bool:
    if not isinstance(text, str) or not text.strip():
        return False
    if NON_LATIN_PATTERN.search(text):
        return False

    return bool(re.search(r"[a-zA-Z]", text))

amazon_df = amazon_df[amazon_df["text"].apply(is_strictly_latin)].copy()

EMOJI_PATTERN = re.compile(
    "["
    "\U0001f600-\U0001f64f"
    "\U0001f300-\U0001f5ff"
    "\U0001f680-\U0001f6ff"
    "\U0001f1e0-\U0001f1ff"
    "\U00002702-\U000027b0"
    "\U000024c2-\U0001f251"
    "]+",
    flags=re.UNICODE,
)

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = EMOJI_PATTERN.sub("", text)
    return re.sub(r"\s+", " ", text).strip()

amazon_df["cleaned_text"] = amazon_df["text"].apply(clean_text)

def preprocess_pipeline(text: str) -> list[str]:
    if not isinstance(text, str):
        return []
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = text.translate(str.maketrans("", "", string.punctuation))
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [w for w in tokens if w not in stop_words]
    # Stem
    return [stemmer.stem(w) for w in tokens]

amazon_df["preprocessed_tokens"] = amazon_df["cleaned_text"].apply(
    preprocess_pipeline
)
amazon_df["preprocessed_text"] = amazon_df["preprocessed_tokens"].apply(
    lambda t: " ".join(t)
)

amazon_df.head(5)

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,cleaned_text,preprocessed_tokens,preprocessed_text
321,615,AmazonHelp,False,Tue Oct 31 22:29:00 +0000 2017,@115820 I'm sorry we've let you down! Without ...,616,617.0,@115820 I'm sorry we've let you down! Without ...,"[115820, im, sorri, weve, let, without, provid...",115820 im sorri weve let without provid person...
322,616,115820,True,Tue Oct 31 23:22:08 +0000 2017,@AmazonHelp 3 different people have given 3 di...,618,615.0,@AmazonHelp 3 different people have given 3 di...,"[amazonhelp, 3, differ, peopl, given, 3, diffe...",amazonhelp 3 differ peopl given 3 differ answe...
323,618,AmazonHelp,False,Tue Oct 31 23:28:00 +0000 2017,@115820 We'd like to take a further look into ...,619,616.0,@115820 We'd like to take a further look into ...,"[115820, wed, like, take, look, pleas, reach, ...",115820 wed like take look pleas reach us phone...
325,617,115820,True,Tue Oct 31 22:16:32 +0000 2017,Way to drop the ball on customer service @1158...,615,NaN,Way to drop the ball on customer service @1158...,"[way, drop, ball, custom, servic, 115821, piss...",way drop ball custom servic 115821 piss right
326,620,AmazonHelp,False,Tue Oct 31 22:28:34 +0000 2017,@115822 I am unable to affect your account via...,NaN,621.0,@115822 I am unable to affect your account via...,"[115822, unabl, affect, account, via, twitter,...",115822 unabl affect account via twitter real t...


Modelling : Logistic Regression

In [69]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

logreg_pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),  # unigrams + bigrams
                min_df=3,  # prune ultra-rare typos
                max_features=10000,  # maintain high speed & low RAM
                sublinear_tf=True,  # 1 + log(tf) scaling
                lowercase=False,
            ),
        ),
        (
            "clf",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",  # adjusts weights for imbalanced intents
                C=1.0,
                solver="lbfgs",
                random_state=42,
            ),
        ),
    ]
)

Applying Labels for Logistic Regression

In [70]:
customer_df = amazon_df[amazon_df["inbound"] == True].copy()

def assign_intent(text: str) -> str:
    t = str(text).lower()

    if re.search(
        r"(where|track|deliver|late|delay|packag|courier|ship|status|arriv|dispatch)",
        t,
    ):
        return "order_tracking"
    elif re.search(r"(refund|cancel|return|money back|return policy)", t):
        return "refund_cancellation"
    elif re.search(
        r"(damag|broken|crack|faulty|defect|wrong item|miss|scratch|tear)", t
    ):
        return "damaged_defective"
    elif re.search(
        r"(login|password|otp|account|sign in|hack|prime membership|subscript)",
        t,
    ):
        return "account_access"
    elif re.search(
        r"(charg|bill|overcharg|deduct|bank|payment|card|transact|invoice)", t
    ):
        return "billing_charge"
    else:
        return "general_inquiry"

# Apply labels
customer_df["intent"] = customer_df["cleaned_text"].apply(assign_intent)

print("Intent breakdown:")
print(customer_df["intent"].value_counts())

Intent breakdown:
intent
general_inquiry        82474
order_tracking         45197
refund_cancellation     8199
account_access          4429
billing_charge          3465
damaged_defective       1706
Name: count, dtype: int64


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

X = customer_df["preprocessed_text"]
y = customer_df["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

logreg_pipeline.fit(X_train, y_train)

y_pred = logreg_pipeline.predict(X_test)
print("\n--- Model Evaluation Report ---")
print(classification_report(y_test, y_pred))

Training Logistic Regression pipeline...

--- Model Evaluation Report ---
                     precision    recall  f1-score   support

     account_access       0.89      0.96      0.93       886
     billing_charge       0.81      0.93      0.87       693
  damaged_defective       0.78      0.87      0.82       341
    general_inquiry       0.96      0.99      0.97     16495
     order_tracking       1.00      0.90      0.95      9039
refund_cancellation       0.89      0.95      0.92      1640

           accuracy                           0.96     29094
          macro avg       0.89      0.93      0.91     29094
       weighted avg       0.96      0.96      0.96     29094



In [72]:
# Filter inbound customer tweets that have real text
candidates = amazon_df[
    (amazon_df["inbound"] == True)
    & (amazon_df["cleaned_text"].str.split().str.len() >= 6)
].copy()

# Sample 15 distinct tweets
unseen_eval = (
    candidates.sample(n=15, random_state=101)[
        ["tweet_id", "text", "cleaned_text", "preprocessed_text"]
    ]
    .reset_index(drop=True)
    .copy()
)

In [73]:
golden_test_data = [
    {
        "text": "@AmazonHelp 3 different people have given 3 different answers for a query on an order placed. This is ridiculous.",
        "true_intent": "general_inquiry",
    },  # Frustration/complaint without order keywords
    {
        "text": "@AmazonHelp I want my amazon payments account CLOSED permanently.",
        "true_intent": "account_access",
    },
    {
        "text": "@AmazonHelp my package was ‘accidentally’ opened..... what do I do",
        "true_intent": "damaged_defective",
    },
    {
        "text": "@AmazonHelp Yeah this is crazy we’re literally just sitting here refreshing tracking pages all day",
        "true_intent": "order_tracking",
    },
    {
        "text": "@AmazonHelp you took money from my bank account without my authorization for prime!",
        "true_intent": "billing_charge",
    },
    {
        "text": "@AmazonHelp I ordered a blue sweater and you guys sent me a coffee mug instead???",
        "true_intent": "damaged_defective",
    },  # Doesn't say 'broken' or 'damage'
    {
        "text": "@AmazonHelp Can I get my money back if the seller never responded?",
        "true_intent": "refund_cancellation",
    },
    {
        "text": "@AmazonHelp Your driver threw the box over my fence in the pouring rain.",
        "true_intent": "order_tracking",
    },  # Contextual delivery failure
    {
        "text": "@AmazonHelp I forgot my password and my 2FA phone number is no longer active.",
        "true_intent": "account_access",
    },
    {
        "text": "@AmazonHelp Why was I billed twice for the exact same order yesterday?",
        "true_intent": "billing_charge",
    },
    {
        "text": "@AmazonHelp Where is my stuff? The app says handed to resident but nobody came.",
        "true_intent": "order_tracking",
    },
    {
        "text": "@AmazonHelp I would like to stop my recurring subscription before next month.",
        "true_intent": "refund_cancellation",
    },  # Doesn't explicitly say 'cancel'
    {
        "text": "@AmazonHelp Is there any way to change the delivery address before it leaves the warehouse?",
        "true_intent": "order_tracking",
    },
    {
        "text": "@AmazonHelp The seal on the bottle was broken and liquid spilled all over the box.",
        "true_intent": "damaged_defective",
    },
    {
        "text": "@AmazonHelp Who can I speak to regarding a partnership with your fulfillment centers?",
        "true_intent": "general_inquiry",
    },
]

eval_benchmark_df = pd.DataFrame(golden_test_data)

In [74]:
# 1. Preprocess the unseen benchmark
eval_benchmark_df["preprocessed_text"] = eval_benchmark_df["text"].apply(
    lambda t: " ".join(preprocess_pipeline(clean_text(t)))
)

# 2. Predict with Logistic Regression
eval_benchmark_df["predicted_intent"] = logreg_pipeline.predict(
    eval_benchmark_df["preprocessed_text"]
)
eval_benchmark_df["is_correct"] = (
    eval_benchmark_df["predicted_intent"] == eval_benchmark_df["true_intent"]
)

# 3. View the reality check
print(
    f"True Benchmark Accuracy: {eval_benchmark_df['is_correct'].mean():.1%}\n"
)

# Display errors
misclassified = eval_benchmark_df[~eval_benchmark_df["is_correct"]]
print(f"Total Errors: {len(misclassified)} / {len(eval_benchmark_df)}")
misclassified[["text", "true_intent", "predicted_intent"]]

True Benchmark Accuracy: 60.0%

Total Errors: 6 / 15


,text,true_intent,predicted_intent
2,@AmazonHelp my package was ‘accidentally’ open...,damaged_defective,order_tracking
4,@AmazonHelp you took money from my bank accoun...,billing_charge,account_access
5,@AmazonHelp I ordered a blue sweater and you g...,damaged_defective,general_inquiry
7,@AmazonHelp Your driver threw the box over my ...,order_tracking,general_inquiry
10,@AmazonHelp Where is my stuff? The app says ha...,order_tracking,general_inquiry
11,@AmazonHelp I would like to stop my recurring ...,refund_cancellation,account_access


In [75]:
print("=== FAILED PREDICTIONS ANALYSIS ===")
for idx, row in misclassified.iterrows():
    print(f"Tweet: \"{row['text']}\"")
    print(f"  • True Intent:      {row['true_intent']}")
    print(f"  • Model Prediction: {row['predicted_intent']}")
    print("-" * 65)

=== FAILED PREDICTIONS ANALYSIS ===
Tweet: "@AmazonHelp my package was ‘accidentally’ opened..... what do I do"
  • True Intent:      damaged_defective
  • Model Prediction: order_tracking
-----------------------------------------------------------------
Tweet: "@AmazonHelp you took money from my bank account without my authorization for prime!"
  • True Intent:      billing_charge
  • Model Prediction: account_access
-----------------------------------------------------------------
Tweet: "@AmazonHelp I ordered a blue sweater and you guys sent me a coffee mug instead???"
  • True Intent:      damaged_defective
  • Model Prediction: general_inquiry
-----------------------------------------------------------------
Tweet: "@AmazonHelp Your driver threw the box over my fence in the pouring rain."
  • True Intent:      order_tracking
  • Model Prediction: general_inquiry
-----------------------------------------------------------------
Tweet: "@AmazonHelp Where is my stuff? The app says ha

In [ ]:
import json
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

client = genai.Client(api_key="")


class SingleItem(BaseModel):
    id: int
    predicted_intent: str = Field(
        description="One of: order_tracking, refund_cancellation, damaged_defective, account_access, billing_charge, general_inquiry"
    )


class BatchResponse(BaseModel):
    results: list[SingleItem]


#Format all 15 tweets with IDs into a single text block
tweet_payload = "\n".join(
    [
        f"[{i}] {text}"
        for i, text in enumerate(eval_benchmark_df["text"])
    ]
)

prompt = f"""
You are an automated support routing agent for AmazonHelp.
Classify each numbered tweet below into exactly ONE intent category:
- order_tracking: Delivery delay, where is package, shipping status, driver issues
- refund_cancellation: Want money back, cancel an order or subscription
- damaged_defective: Item arrived broken, crushed, seal opened, or wrong item sent
- account_access: Login trouble, OTP, password, account closing, 2FA
- billing_charge: Overbilled, unrecognized payment, card deducted twice
- general_inquiry: General feedback, partnership, praise, generic service complaints

Tweets to classify:
{tweet_payload}
"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=BatchResponse,
        temperature=0.0,
    ),
)

parsed_output = json.loads(response.text)["results"]
pred_map = {item["id"]: item["predicted_intent"] for item in parsed_output}

eval_benchmark_df["llm_prediction"] = [
    pred_map.get(i, "general_inquiry") for i in range(len(eval_benchmark_df))
]
eval_benchmark_df["llm_correct"] = (
    eval_benchmark_df["llm_prediction"] == eval_benchmark_df["true_intent"]
)

print("=== BASELINE VS LLM AGENT ACCURACY ===")
print(
    f"Classical Baseline (Logistic Regression): {eval_benchmark_df['is_correct'].mean():.1%}"
)
print(
    f"LLM Agent (Gemini 2.5 Flash):             {eval_benchmark_df['llm_correct'].mean():.1%}"
)

=== BASELINE VS LLM AGENT ACCURACY ===
Classical Baseline (Logistic Regression): 60.0%
LLM Agent (Gemini 2.5 Flash):             100.0%


Two-Tier Hybrid Routing Function  

It runs the query through Logistic Regression first. If the model is confident ($\ge 0.70$), it uses that result for free. If the confidence is low ($< 0.70$, which is where classical ML usually fails), it sends the query to Gemini to resolve the ambiguity.

In [81]:
import numpy as np


def classify_customer_tweet(tweet_text: str, confidence_cutoff: float = 0.70):
    # 1. Preprocess the incoming tweet
    cleaned = clean_text(tweet_text)
    processed = " ".join(preprocess_pipeline(cleaned))

    # 2. Get Logistic Regression prediction and confidence
    probs = logreg_pipeline.predict_proba([processed])[0]
    best_idx = np.argmax(probs)
    logreg_intent = logreg_pipeline.classes_[best_idx]
    confidence = probs[best_idx]

    # 3. Decision rule:
    # If Logistic Regression is confident, return it directly (Fast & Free)
    if confidence >= confidence_cutoff:
        return {
            "tweet": tweet_text,
            "handler": "Logistic Regression (Tier 1)",
            "intent": logreg_intent,
            "confidence": round(float(confidence), 2),
        }

    # If confidence is low, fall back to Gemini
    llm_prompt = f"""
    Classify this Amazon customer tweet into exactly one intent:
    (order_tracking, refund_cancellation, damaged_defective, account_access, billing_charge, general_inquiry)

    Tweet: "{tweet_text}"
    Return ONLY the category name.
    """
    response = client.models.generate_content(
        model="gemini-2.5-flash", contents=llm_prompt
    )
    llm_intent = response.text.strip().lower()

    return {
        "tweet": tweet_text,
        "handler": "Gemini 2.5 Flash (Tier 2 Fallback)",
        "intent": llm_intent,
        "confidence": round(float(confidence), 2),
    }

In [82]:
test_queries = [
    # 1. Clear query (High confidence -> Logistic Regression handles it)
    "Where is my package? It was supposed to be delivered yesterday.",
    # 2. Ambiguous query (Low confidence -> Gemini catches it)
    "I ordered a blue sweater and you guys sent me a coffee mug instead???",
    # 3. Frustration / Cancellation query
    "I want to stop my recurring prime subscription right now.",
]

for query in test_queries:
    result = classify_customer_tweet(query)
    print(f"Tweet:    \"{result['tweet']}\"")
    print(
        f"Routed:   {result['handler']} (Confidence: {result['confidence']})"
    )
    print(f"Intent:   {result['intent']}")
    print("-" * 60)

Tweet:    "Where is my package? It was supposed to be delivered yesterday."
Routed:   Logistic Regression (Tier 1) (Confidence: 1.0)
Intent:   order_tracking
------------------------------------------------------------
Tweet:    "I ordered a blue sweater and you guys sent me a coffee mug instead???"
Routed:   Logistic Regression (Tier 1) (Confidence: 0.76)
Intent:   general_inquiry
------------------------------------------------------------
Tweet:    "I want to stop my recurring prime subscription right now."
Routed:   Logistic Regression (Tier 1) (Confidence: 1.0)
Intent:   account_access
------------------------------------------------------------


In [83]:
def amazon_agent(tweet: str):
    prompt = f"""
    You are an Amazon Twitter support agent.
    Analyze this customer tweet: "{tweet}"

    Give your answer in EXACTLY this format (do not add extra text):
    Intent: (choose one: order_tracking, refund_cancellation, damaged_defective, account_access, billing_charge, general_inquiry)
    Decision: (choose one: AUTO_HANDLE or ESCALATE_TO_HUMAN)
    Reason: (one short sentence why)
    Reply: (short polite Amazon tweet reply ending with ^AMZ)
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    return response.text

In [84]:
test_tweets = [
    "Where is my package? It was supposed to arrive yesterday.",
    "I ordered a blue sweater and you guys sent me a coffee mug instead???",
    "You took $100 from my bank account! Refund this now or I am calling my lawyer!"
]

for tweet in test_tweets:
    print("CUSTOMER:", tweet)
    print(amazon_agent(tweet))
    print("-" * 50)

CUSTOMER: Where is my package? It was supposed to arrive yesterday.
Intent: order_tracking
Decision: AUTO_HANDLE
Reason: The initial tweet can provide guidance to track the package or direct to a secure channel for further assistance.
Reply: Oh no! We're sorry your package hasn't arrived. Please DM us your order number so we can look into this for you. ^AMZ
--------------------------------------------------
CUSTOMER: I ordered a blue sweater and you guys sent me a coffee mug instead???
Intent: refund_cancellation
Decision: ESCALATE_TO_HUMAN
Reason: A human agent needs to verify order details and arrange for the correct item or a refund.
Reply: Oh no, that's definitely not what you ordered! We're sorry for this mix-up. Please DM us with your order ID so we can get this fixed for you. ^AMZ
--------------------------------------------------
CUSTOMER: You took $100 from my bank account! Refund this now or I am calling my lawyer!
Intent: billing_charge
Decision: ESCALATE_TO_HUMAN
Reason: Cu

In [ ]:
import joblib
joblib.dump(logreg_pipeline, "logreg_pipeline.joblib")
print("Saved model to logreg_pipeline.joblib")

Saved model to logreg_pipeline.joblib
